# Colab Quickstart
This notebook shows how to run the `alpha_data` trainer on Google Colab.
Place your datasets under `/content/data/` (or mount Google Drive) and update paths below.

In [15]:
%%bash
# Clone or update the GitHub repo into /content/colab-testing
if [ -d /content/colab-testing/.git ]; then
  echo 'Repository already exists at /content/colab-testing, pulling latest changes'
  cd /content/colab-testing
  git pull
else
  git clone --depth 1 https://github.com/mohamedIzoughne/gender-age-prediction-mobileNetv3-model.git /content/colab-testing
  cd /content/colab-testing
fi

# Install requirements if present
if [ -f requirements.txt ]; then
  pip install -r requirements.txt
else
  pip install torch torchvision pandas pillow
fi

Repository already exists at /content/colab-testing, pulling latest changes
Updating a5e2185..b6e8ec3
Fast-forward
 .gitignore                        |  2 +-
 notebooks/colab_quickstart.ipynb  | 67 +++++++++++++++++++++++++++++++++++----
 src/__init__.py                   |  2 ++
 src/alpha_data/models/__init__.py | 20 ++++++++++++
 src/alpha_data/models/model.py    | 47 +++++++++++++++++++++++++++
 src/alpha_data/train.py           |  8 +++--
 6 files changed, 137 insertions(+), 9 deletions(-)
 create mode 100644 src/__init__.py
 create mode 100644 src/alpha_data/models/__init__.py
 create mode 100644 src/alpha_data/models/model.py


From https://github.com/mohamedIzoughne/gender-age-prediction-mobileNetv3-model
   a5e2185..b6e8ec3  main       -> origin/main


In [16]:
# Prepare environment: set project directory, add to PYTHONPATH and install requirements
import sys, os, subprocess

# Adjust this to where you upload or mount the repository in Colab
project_dir = '/content/colab-testing'  # change if you placed the repo elsewhere

if os.path.exists(project_dir):
    os.chdir(project_dir)
    if project_dir not in sys.path:
        sys.path.insert(0, project_dir)
    print('Working directory set to', project_dir)
    print('sys.path updated.')
else:
    print('project_dir not found:', project_dir)

# Install requirements if present in the repo root
req = os.path.join(project_dir, 'requirements.txt')
if os.path.exists(req):
    print('Installing dependencies from', req)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', req])
else:
    print('requirements.txt not found at', req, '\nIf you mounted Google Drive, set project_dir to the mounted path.')

Working directory set to /content/colab-testing
sys.path updated.
Installing dependencies from /content/colab-testing/requirements.txt


In [11]:
# Inspect the cloned repository and important paths
print('Current working directory:')
!pwd

print('\nTop-level files:')
!ls -la /content/colab-testing

print('\nShow src package:')
!ls -la /content/colab-testing/src

print('\nShow alpha_data package:')
!ls -la /content/colab-testing/src/alpha_data

print('\nQuick find (depth 2):')
!find /content/colab-testing -maxdepth 2 -type d -printf '%p\n' | sed -n '1,200p'

Current working directory:
/content/colab-testing

Top-level files:
total 36
drwxr-xr-x 5 root root 4096 Jul 19 18:47 .
drwxr-xr-x 1 root root 4096 Jul 19 18:47 ..
-rw-r--r-- 1 root root 1273 Jul 19 18:47 experiment.ipynb
drwxr-xr-x 8 root root 4096 Jul 19 18:47 .git
-rw-r--r-- 1 root root   72 Jul 19 18:47 .gitignore
drwxr-xr-x 2 root root 4096 Jul 19 18:47 notebooks
-rw-r--r-- 1 root root 2423 Jul 19 18:47 README.md
-rw-r--r-- 1 root root   47 Jul 19 18:47 requirements.txt
drwxr-xr-x 3 root root 4096 Jul 19 18:47 src

Show src package:
total 12
drwxr-xr-x 3 root root 4096 Jul 19 18:47 .
drwxr-xr-x 5 root root 4096 Jul 19 18:47 ..
drwxr-xr-x 4 root root 4096 Jul 19 18:48 alpha_data

Show alpha_data package:
total 40
drwxr-xr-x 4 root root 4096 Jul 19 18:48 .
drwxr-xr-x 3 root root 4096 Jul 19 18:47 ..
-rw-r--r-- 1 root root  334 Jul 19 18:47 config.py
-rw-r--r-- 1 root root 1820 Jul 19 18:47 data.py
drwxr-xr-x 2 root root 4096 Jul 19 18:47 datasets
-rw-r--r-- 1 root root  109 Jul 19 1

## Two-Stage Training (Recommended)

### Stage 1: Train Heads Only (Backbone Frozen)
Upload or copy UTKFace images into `/content/data/UTKFace`. Train the age/gender prediction heads while keeping the backbone frozen for fast convergence.


In [21]:
# Run Stage 1 training (heads only, frozen backbone)
!PYTHONPATH=/content/colab-testing python -m src.alpha_data.train --dataset utkface --data-dir /content/data/UTKFace --model alpha_mobilenetv3 --stage 1 --epochs 30 --batch-size 32

<frozen runpy>:128: RuntimeWarning: 'src.alpha_data.train' found in sys.modules after import of package 'src.alpha_data', but prior to execution of 'src.alpha_data.train'; this may result in unpredictable behaviour
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth
100% 21.1M/21.1M [00:00<00:00, 133MB/s]
Stage 1: Training heads only (backbone frozen)
^C


In [ ]:
nvidia-smi

### Stage 2: Fine-tune Backbone (Transfer Learning)
After Stage 1 completes, fine-tune the backbone to avoid catastrophic forgetting. The optimizer automatically uses 10% of the LR and adds light regularization.


In [ ]:
# Run Stage 2 training (fine-tune backbone, lower LR for stability)
!PYTHONPATH=/content/colab-testing python -m src.alpha_data.train --dataset utkface --data-dir /content/data/UTKFace --model alpha_mobilenetv3 --stage 2 --epochs 10 --batch-size 32 --lr 1e-4

## Track Results

All experiment metrics are logged to CSV (`results.csv` by default). Use `--results-csv` to specify a custom file.


In [ ]:
import pandas as pd

# Load results CSV (tracks all runs including both stages)
results_df = pd.read_csv('results.csv')

print("Latest epochs from both stages:")
print(results_df[['epoch', 'stage', 'val_age_acc', 'val_gender_acc', 'val_loss']].tail(20))

# Compare stage performance: average metrics per stage
stage_comparison = results_df.groupby(['stage', 'dataset'])[['val_age_acc', 'val_gender_acc']].agg(['mean', 'min', 'max'])
print("\n=== Stage Comparison ===")
print(stage_comparison)

# Show improvement from stage 1 to stage 2
stage1_last = results_df[results_df['stage'] == 1].iloc[-1] if (results_df['stage'] == 1).any() else None
stage2_last = results_df[results_df['stage'] == 2].iloc[-1] if (results_df['stage'] == 2).any() else None

if stage1_last is not None and stage2_last is not None:
    print(f"\nStage 1 (final): Age Acc={stage1_last['val_age_acc']}%, Gender Acc={stage1_last['val_gender_acc']}%")
    print(f"Stage 2 (final): Age Acc={stage2_last['val_age_acc']}%, Gender Acc={stage2_last['val_gender_acc']}%")
